In [4]:
 # Generate Dataset
import numpy as np
import matplotlib.pyplot as plt

N = 3000
x1 = np.random.uniform(-2, 2, N)
x2 = np.random.uniform(-2, 2, N)

X = np.column_stack((x1,x2))
y = ((x1 ** 2 + x2 ** 2) > 1.5).astype(int).reshape(-1,1)

In [5]:
# Train/Validation/Test Split

idx = np.random.permutation(N)
train_end = int(0.7 * N)
val_end = int(0.85 * N)

X_train = X[idx[:train_end]]
y_train = y[idx[:train_end]]

X_val = X[idx[train_end:val_end]]
Y_val = y[idx[train_end:val_end]]

X_test = X[idx[val_end:]]
Y_test = y[idx[val_end:]]


Task 1A: Build Three Architectures

You must build:

1. 2-layer network

2. 5-layer network

3. 10-layer network

Constraints:

≥
 4 neurons per hidden layer

1 output neuron (sigmoid)

Fully connected

Hidden activations:

Sigmoid

ReLU

In [6]:
# Activations
def sigmoid(z):
  return 1/(1+np.exp(-z))

def sigmoid_derive(z):
  s = sigmoid(z)
  return s*(1-s)

def relu(z):
  return np.maximum(0,z)

def relu_derive(z):
  return (z>0).astype(float)

In [7]:
# LOSS(Binary Cross Entropy with SStability)
def bce_loss(y,y_hat):
  eps = 1e-8
  y_hat = np.clip(y_hat,eps,1-eps)
  return -np.mean(y*np.log(y_hat) + (1 - y)*np.log(1-y_hat))

def accuracy(y, y_hat):
  preds = (y_hat >= 0.5).astype(int)
  return np.mean(preds == y)

In [8]:
# Task 1B": Parameter Calculations

# Parameter Initializations
def init_network(layer_sizes):
    params = {}

    for l in range(1, len(layer_sizes)):
        in_dim = layer_sizes[l-1]
        out_dim = layer_sizes[l]

        params["W"+str(l)] = np.random.randn(in_dim, out_dim) * 0.1
        params["b"+str(l)] = np.zeros((1, out_dim))

    return params

In [9]:
# Forward Pass
def forward(X, params, activation_name):
  cache = {}
  A = X
  L = len(params)//2

  for l in range(1, L):
    Z = A @ params["W"+str(l)] + params["b"+str(l)]
    cache["Z"+str(l)] = Z

    if activation_name == "relu":
      A = relu(Z)
    else:
      A = sigmoid(Z)

    cache["A" + str(l)] = A

  # Output Layer (Sigmoid always)
  ZL = A @ params["W" + str(L)] + params["b"+str(L)]
  AL = sigmoid(ZL)

  cache["Z"+str(L)] = ZL
  cache["A"+str(L)] = AL

  return AL, cache

In [10]:
# BackPropagation
def backward(X, y, params, cache, activation_name):
  grads = {}
  L = len(params)//2
  m = X.shape[0]

  AL = cache["A"+str(L)]
  dZ = AL - y

  for l in reversed(range(1, L+1)):
    A_prev = X if l==1 else cache["A"+str(l-1)]

    grads["dW"+str(l)] = A_prev.T @ dZ / m
    grads["db"+str(l)] = np.sum(dZ, axis=0, keepdims=True) / m

    if l>1:
      dA_prev = dZ @ params["W"+str(l)].T
      Z_prev = cache["Z"+str(l-1)]

      if activation_name == "relu":
        dZ = dA_prev * relu_derive(Z_prev)
      else:
        dZ = dA_prev * sigmoid_derive(Z_prev)

  return grads


In [11]:
# OPTIMIZERS
def update_sgd(params, grads, lr):
    L = len(params)//2
    for l in range(1, L+1):
        params["W"+str(l)] -= lr * grads["dW"+str(l)]
        params["b"+str(l)] -= lr * grads["db"+str(l)]
    return params

In [12]:
# Momentum
def init_velocity(params):
    velocity = {}
    L = len(params)//2
    for l in range(1, L+1):
        velocity["dW"+str(l)] = np.zeros_like(params["W"+str(l)])
        velocity["db"+str(l)] = np.zeros_like(params["b"+str(l)])
    return velocity

def update_momentum(params, grads, velocity, lr, beta=0.9):
    L = len(params)//2
    for l in range(1, L+1):
        velocity["dW"+str(l)] = beta*velocity["dW"+str(l)] + lr*grads["dW"+str(l)]
        velocity["db"+str(l)] = beta*velocity["db"+str(l)] + lr*grads["db"+str(l)]

        params["W"+str(l)] -= velocity["dW"+str(l)]
        params["b"+str(l)] -= velocity["db"+str(l)]
    return params, velocity

In [30]:
def train_model(X_train, y_train,
                X_val, y_val,
                layer_sizes,
                activation_name,
                optimizer="sgd",
                epochs=200,
                lr=0.01):

    params = init_network(layer_sizes)

    if optimizer == "momentum":
        velocity = init_velocity(params)

    history = {"train_loss":[], "val_loss":[]}

    for epoch in range(epochs):

        y_hat, cache = forward(X_train, params, activation_name)

        loss = bce_loss(y_train, y_hat)

        grads = backward(X_train, y_train, params, cache, activation_name)

        if optimizer == "sgd":
            params = update_sgd(params, grads, lr)
        else:
            params, velocity = update_momentum(params, grads, velocity, lr)

        val_hat,_ = forward(X_val, params, activation_name)
        val_loss = bce_loss(y_val, val_hat)

        history["train_loss"].append(loss)
        history["val_loss"].append(val_loss)

    return params, history
    params, hist = train_model(
    X_train, y_train,
    X_val, y_val,
    layers_5,
    "relu",
    optimizer="momentum"
)

In [28]:
params, hist = train_model(
    X_train, y_train,
    X_val, y_val,
    layers_5,
    "relu",
    optimizer="momentum"
)

NameError: name 'y_val' is not defined

In [31]:
# Build Arichitectures

# 2-layer
layers_2 = [2, 8, 1]

# 5 layer
layers_5 = [2, 8, 8, 8, 8, 1]

# 10 layer
layers_10 = [2] + [8]*9 + [1]

params, hist = train_model(layers_5, "relu", "momentum")

TypeError: train_model() missing 3 required positional arguments: 'y_val', 'layer_sizes', and 'activation_name'

PART 2: CNN FROM SCRATCH

In [ ]:
# Convolution forward

def conv_forward(X, K):
    N,H,W = X.shape
    F = K.shape[0]
    out = np.zeros((N, H-F+1, W-F+1))

    for n in range(N):
        for i in range(H-F+1):
            for j in range(W-F+1):
                region = X[n,i:i+F,j:j+F]
                out[n,i,j] = np.sum(region * K)

    return out


In [ ]:
# Relu
def relu(Z):
    return np.maximum(0,Z)

def relu_deriv(Z):
    return (Z > 0).astype(float)

In [ ]:
# Max Pooling
def max_pool(X):
    N,H,W = X.shape
    out = np.zeros((N,H//2,W//2))

    for n in range(N):
        for i in range(0,H,2):
            for j in range(0,W,2):
                out[n,i//2,j//2] = np.max(X[n,i:i+2,j:j+2])

    return out

In [ ]:
# Flatten layer
def flatten(X):
    return X.reshape(X.shape[0], -1)

In [ ]:
# Initialize CNN Parameters
def init_cnn():
    params = {}
    params["K"] = np.random.randn(3,3) * 0.1
    params["W"] = np.random.randn(9,1) * 0.1
    params["b"] = np.zeros((1,1))
    return params

In [ ]:
# Forward Pass
def cnn_forward(X, params):

    conv = conv_forward(X, params["K"])
    relu_out = relu(conv)
    pool = max_pool(relu_out)
    flat = flatten(pool)

    z = flat @ params["W"] + params["b"]
    y_hat = 1/(1+np.exp(-z))

    cache = {
        "conv":conv,
        "relu":relu_out,
        "pool":pool,
        "flat":flat,
        "y_hat":y_hat
    }

    return y_hat, cache

In [ ]:
# Backpropagation
def dense_backward(y, params, cache):
    m = y.shape[0]
    dz = cache["y_hat"] - y

    dW = cache["flat"].T @ dz / m
    db = np.sum(dz,axis=0,keepdims=True)/m
    dflat = dz @ params["W"].T

    return dW, db, dflat

In [ ]:
# Unflatten
def unflatten(dflat):
    return dflat.reshape(-1,3,3)

In [ ]:
def max_pool_backward(dpool, relu_out):
    N,H,W = relu_out.shape
    dconv = np.zeros_like(relu_out)

    for n in range(N):
        for i in range(0,H,2):
            for j in range(0,W,2):
                region = relu_out[n,i:i+2,j:j+2]
                max_val = np.max(region)
                for x in range(2):
                    for y in range(2):
                        if region[x,y]==max_val:
                            dconv[n,i+x,j+y] = dpool[n,i//2,j//2]
    return dconv

In [ ]:
def conv_backward(dconv, X, K):
    N,H,W = X.shape
    F = K.shape[0]
    dK = np.zeros_like(K)

    for n in range(N):
        for i in range(H-F+1):
            for j in range(W-F+1):
                region = X[n,i:i+F,j:j+F]
                dK += region * dconv[n,i,j]

    return dK/N

In [ ]:
# Update parameters
def update(params, grads, lr):
    params["W"] -= lr*grads["dW"]
    params["b"] -= lr*grads["db"]
    params["K"] -= lr*grads["dK"]
    return params

In [ ]:
# Training Loop
def train_cnn(X_train,y_train,X_val,y_val,epochs=50,lr=0.01):

    params = init_cnn()

    for epoch in range(epochs):

        y_hat,cache = cnn_forward(X_train,params)

        loss = bce_loss(y_train,y_hat)

        dW,db,dflat = dense_backward(y_train,params,cache)
        dpool = unflatten(dflat)
        dconv = max_pool_backward(dpool,cache["relu"])
        dK = conv_backward(dconv,X_train,params["K"])

        grads = {"dW":dW,"db":db,"dK":dK}
        params = update(params,grads,lr)

        print("Epoch",epoch,"Loss:",loss)

    return params

In [ ]:
import numpy as np

def generate_image_data(n_samples=3000):

    X = np.zeros((n_samples, 8, 8))
    y = np.zeros((n_samples,1))

    for i in range(n_samples):

        img = np.zeros((8,8))

        if np.random.rand() > 0.5:
            # Class 1: vertical line
            col = np.random.randint(2,6)
            img[:,col] = 1
            y[i] = 1
        else:
            # Class 0: horizontal line
            row = np.random.randint(2,6)
            img[row,:] = 1
            y[i] = 0

        X[i] = img

    return X, y

In [ ]:
X_img, y_img = generate_image_data(3000)

In [15]:
X_train_img = X_img[:2100]
y_train_img = y_img[:2100]

X_val_img = X_img[2100:2550]
y_val_img = y_img[2100:2550]

X_test_img = X_img[2550:]
y_test_img = y_img[2550:]

NameError: name 'X_img' is not defined

In [ ]:
print(X_train_img.shape)
print(y_train_img.shape)

In [ ]:
params_pool = train_cnn(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    epochs=50,
    lr=0.01
)

In [ ]:
# Initialize CNN(Without Pool)
def init_cnn_no_pool():
    params = {}
    params["K"] = np.random.randn(3,3) * 0.1
    params["W"] = np.random.randn(36,1) * 0.1
    params["b"] = np.zeros((1,1))
    return params

In [ ]:
# Forward (No Pool)
def cnn_forward_no_pool(X, params):

    conv = conv_forward(X, params["K"])
    relu_out = relu(conv)

    flat = flatten(relu_out)

    z = flat @ params["W"] + params["b"]
    y_hat = 1/(1+np.exp(-z))

    cache = {
        "conv":conv,
        "relu":relu_out,
        "flat":flat,
        "y_hat":y_hat
    }

    return y_hat, cache

In [16]:
# BAckward(No Pool)
def cnn_backward_no_pool(X, y, params, cache):

    m = y.shape[0]

    dz = cache["y_hat"] - y

    dW = cache["flat"].T @ dz / m
    db = np.sum(dz,axis=0,keepdims=True)/m

    dflat = dz @ params["W"].T
    dconv = dflat.reshape(-1,6,6)

    dconv_relu = dconv * relu_deriv(cache["conv"])

    dK = conv_backward(dconv_relu, X, params["K"])

    grads = {"dW":dW,"db":db,"dK":dK}
    return grads

In [17]:
# Train(No Pool)
def train_cnn_no_pool(X_train,y_train,epochs=50,lr=0.01):

    params = init_cnn_no_pool()

    for epoch in range(epochs):

        y_hat,cache = cnn_forward_no_pool(X_train,params)

        loss = bce_loss(y_train,y_hat)

        grads = cnn_backward_no_pool(X_train,y_train,params,cache)

        params["W"] -= lr*grads["dW"]
        params["b"] -= lr*grads["db"]
        params["K"] -= lr*grads["dK"]

        print("Epoch",epoch,"Loss:",loss)

    return params

In [18]:
# Dropout Forward
def dropout_forward(A, drop_rate=0.5):

    mask = (np.random.rand(*A.shape) > drop_rate).astype(float)
    A_drop = A * mask
    A_drop /= (1 - drop_rate)

    return A_drop, mask

In [19]:
# Dropout Backward
def dropout_backward(dA, mask, drop_rate=0.5):
    dA = dA * mask
    dA /= (1 - drop_rate)
    return dA

In [20]:
# CNN Forward with Dropout
def cnn_forward_dropout(X, params, drop_rate=0.5):

    conv = conv_forward(X, params["K"])
    relu_out = relu(conv)
    pool = max_pool(relu_out)
    flat = flatten(pool)

    flat_drop, mask = dropout_forward(flat, drop_rate)

    z = flat_drop @ params["W"] + params["b"]
    y_hat = 1/(1+np.exp(-z))

    cache = {
        "conv":conv,
        "relu":relu_out,
        "pool":pool,
        "flat":flat,
        "flat_drop":flat_drop,
        "mask":mask,
        "y_hat":y_hat
    }

    return y_hat, cache

In [21]:
# CNN Backward with Dropout
def cnn_backward_dropout(X, y, params, cache, drop_rate=0.5):

    m = y.shape[0]

    dz = cache["y_hat"] - y

    dW = cache["flat_drop"].T @ dz / m
    db = np.sum(dz,axis=0,keepdims=True)/m

    dflat = dz @ params["W"].T

    dflat = dropout_backward(dflat, cache["mask"], drop_rate)

    dpool = dflat.reshape(-1,3,3)

    dconv = max_pool_backward(dpool, cache["relu"])

    dconv_relu = dconv * relu_deriv(cache["conv"])

    dK = conv_backward(dconv_relu, X, params["K"])

    grads = {"dW":dW,"db":db,"dK":dK}
    return grads

In [22]:
# Train with Dropout
def train_cnn_dropout(X_train,y_train,epochs=50,lr=0.01,drop_rate=0.5):

    params = init_cnn()

    for epoch in range(epochs):

        y_hat,cache = cnn_forward_dropout(X_train,params,drop_rate)

        loss = bce_loss(y_train,y_hat)

        grads = cnn_backward_dropout(X_train,y_train,params,cache,drop_rate)

        params["W"] -= lr*grads["dW"]
        params["b"] -= lr*grads["db"]
        params["K"] -= lr*grads["dK"]

        print("Epoch",epoch,"Loss:",loss)

    return params

In [24]:
train_pred, _ = cnn_forward(X_train_img, params_pool)
val_pred, _   = cnn_forward(X_val_img, params_pool)
test_pred, _  = cnn_forward(X_test_img, params_pool)

print("Train Acc:", accuracy(y_train_img, train_pred))
print("Val Acc:", accuracy(y_val_img, val_pred))
print("Test Acc:", accuracy(y_test_img, test_pred))

NameError: name 'cnn_forward' is not defined

In [25]:
params_no_pool = train_cnn_no_pool(
    X_train_img, y_train_img,
    epochs=50, lr=0.01
)

NameError: name 'X_train_img' is not defined

In [ ]:
params_dropout = train_cnn_dropout(
    X_train_img, y_train_img,
    epochs=50, lr=0.01, drop_rate=0.5
)

In [ ]:
def evaluate_model(X_train, y_train,
                   X_val, y_val,
                   X_test, y_test,
                   params, forward_fn):

    train_pred, _ = forward_fn(X_train, params)
    val_pred, _   = forward_fn(X_val, params)
    test_pred, _  = forward_fn(X_test, params)

    train_acc = accuracy(y_train, train_pred)
    val_acc   = accuracy(y_val, val_pred)
    test_acc  = accuracy(y_test, test_pred)

    return train_acc, val_acc, test_acc

In [ ]:
train_acc_pool, val_acc_pool, test_acc_pool = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_pool,
    cnn_forward
)

In [ ]:
train_acc_no_pool, val_acc_no_pool, test_acc_no_pool = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_no_pool,
    cnn_forward_no_pool
)

In [ ]:
train_acc_no_pool, val_acc_no_pool, test_acc_no_pool = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_no_pool,
    cnn_forward_no_pool
)

In [ ]:
train_acc_drop, val_acc_drop, test_acc_drop = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_dropout,
    cnn_forward
)

In [ ]:
gap_pool     = train_acc_pool - val_acc_pool
gap_no_pool  = train_acc_no_pool - val_acc_no_pool
gap_drop     = train_acc_drop - val_acc_drop

In [ ]:
print("MODEL PERFORMANCE COMPARISON")
print("--------------------------------------------------")
print("Model           | Train  | Val    | Test   | Gap")
print("--------------------------------------------------")
print(f"With Pooling    | {train_acc_pool:.3f} | {val_acc_pool:.3f} | {test_acc_pool:.3f} | {gap_pool:.3f}")
print(f"No Pooling      | {train_acc_no_pool:.3f} | {val_acc_no_pool:.3f} | {test_acc_no_pool:.3f} | {gap_no_pool:.3f}")
print(f"With Dropout    | {train_acc_drop:.3f} | {val_acc_drop:.3f} | {test_acc_drop:.3f} | {gap_drop:.3f}")

1. Which model generalizes better on test set — dense or CNN?

The CNN model generalizes better than the Dense (MLP) model.

Reason:

CNN with pooling achieved ~73–74% test accuracy.

Dense models (from Part 1) typically perform worse on image-like data because they do not preserve spatial structure.

Why?

Dense layers:

Treat pixels as independent features.

Ignore spatial relationships.

CNN:

Learns local spatial patterns (edges, lines).

Shares weights across space.

Captures structure efficiently.

2. Does pooling improve test performance?

Yes, Pooling improved test accuracy significantly.

Why?

Pooling:

Reduces spatial noise

Makes model translation-invariant

Reduces sensitivity to small pixel shifts

Improves generalization

3. Does dropout reduce validation-test gap?

Slightly, but not significantly.
However:

Accuracy decreased overall (~50%)

Model started underfitting

Why?

Your CNN is small (1 conv + 1 dense).

Dropout is more useful for:

Deep networks

High-capacity models

Strong overfitting cases

In small models, dropout can reduce learning capacity too much.

4. How does parameter count relate to test accuracy?

Parameter count must be balanced.

Too few parameters → underfitting → low test accuracy

Too many parameters → overfitting → poor generalization

CNN uses fewer parameters than dense networks for images because:

Convolution shares weights

Local connectivity reduces parameter count

5. Does CNN scale better with larger image size?

Yes

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(history, title="Loss Curve"):
    plt.figure(figsize=(6,4))
    plt.plot(history["train_loss"], linewidth=2)
    plt.plot(history["val_loss"], linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend(["Train Loss","Validation Loss"])
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_accuracy_comparison():

    models = ["Pooling", "No Pool", "Dropout"]
    train = [train_acc_pool, train_acc_no_pool, train_acc_drop]
    val   = [val_acc_pool, val_acc_no_pool, val_acc_drop]
    test  = [test_acc_pool, test_acc_no_pool, test_acc_drop]

    x = range(len(models))

    plt.figure(figsize=(7,4))
    plt.bar([i-0.2 for i in x], train, width=0.2)
    plt.bar(x, val, width=0.2)
    plt.bar([i+0.2 for i in x], test, width=0.2)

    plt.xticks(x, models)
    plt.ylabel("Accuracy")
    plt.title("Model Accuracy Comparison")
    plt.legend(["Train","Validation","Test"])
    plt.ylim(0,1)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_accuracy_comparison()

In [ ]:
# Overfitting Gap Plot
def plot_overfitting_gap():

    models = ["Pooling", "No Pool", "Dropout"]
    gaps = [
        train_acc_pool - val_acc_pool,
        train_acc_no_pool - val_acc_no_pool,
        train_acc_drop - val_acc_drop
    ]

    plt.figure(figsize=(6,4))
    plt.bar(models, gaps)
    plt.ylabel("Train - Validation Gap")
    plt.title("Overfitting Comparison")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_overfitting_gap()

PART 3: Optimizer Behaviour in CNN

In [ ]:
def update_sgd(params, grads, lr):
    params["W"] -= lr * grads["dW"]
    params["b"] -= lr * grads["db"]
    params["K"] -= lr * grads["dK"]
    return params

In [ ]:
def init_momentum(params):
    v = {}
    for key in params:
        v[key] = np.zeros_like(params[key])
    return v

In [ ]:
# Momentum Update
def update_momentum(params, grads, velocity, lr, beta=0.9):

    velocity["W"] = beta * velocity["W"] + (1-beta) * grads["dW"]
    velocity["b"] = beta * velocity["b"] + (1-beta) * grads["db"]
    velocity["K"] = beta * velocity["K"] + (1-beta) * grads["dK"]

    params["W"] -= lr * velocity["W"]
    params["b"] -= lr * velocity["b"]
    params["K"] -= lr * velocity["K"]

    return params, velocity

In [ ]:
# Intialize Adam
def init_adam(params):
    m = {}
    v = {}
    for key in params:
        m[key] = np.zeros_like(params[key])
        v[key] = np.zeros_like(params[key])
    return m, v

In [ ]:
# Adam Update
def update_adam(params, grads, m, v, lr,
                beta1=0.9, beta2=0.999, eps=1e-8):

    for key, grad_key in zip(["W","b","K"], ["dW","db","dK"]):

        m[key] = beta1*m[key] + (1-beta1)*grads[grad_key]
        v[key] = beta2*v[key] + (1-beta2)*(grads[grad_key]**2)

        params[key] -= lr * m[key] / (np.sqrt(v[key]) + eps)

    return params, m, v

In [ ]:
def cnn_backward(X, y, params, cache):

    m = y.shape[0]

    # Dense backward
    dz = cache["y_hat"] - y

    dW = cache["flat"].T @ dz / m
    db = np.sum(dz, axis=0, keepdims=True) / m

    dflat = dz @ params["W"].T

    # Unflatten
    dpool = dflat.reshape(-1, 3, 3)

    # Pool backward
    dconv = max_pool_backward(dpool, cache["relu"])

    # ReLU backward
    dconv_relu = dconv * relu_deriv(cache["conv"])

    # Conv backward
    dK = conv_backward(dconv_relu, X, params["K"])

    grads = {
        "dW": dW,
        "db": db,
        "dK": dK
    }

    return grads

In [ ]:
# Unified Training Function
def train_cnn_optimizer(X_train, y_train,
                        X_val, y_val,
                        optimizer="sgd",
                        epochs=50, lr=0.01):

    params = init_cnn()

    if optimizer == "momentum":
        velocity = init_momentum(params)

    if optimizer == "adam":
        m, v = init_adam(params)

    history = {"train_loss":[], "val_loss":[]}

    for epoch in range(epochs):

        y_hat, cache = cnn_forward(X_train, params)
        loss = bce_loss(y_train, y_hat)

        grads = cnn_backward(X_train, y_train, params, cache)

        if optimizer == "sgd":
            params = update_sgd(params, grads, lr)

        elif optimizer == "momentum":
            params, velocity = update_momentum(params, grads, velocity, lr)

        elif optimizer == "adam":
            params, m, v = update_adam(params, grads, m, v, lr)

        val_hat,_ = cnn_forward(X_val, params)
        val_loss = bce_loss(y_val, val_hat)

        history["train_loss"].append(loss)
        history["val_loss"].append(val_loss)

        print(f"{optimizer} Epoch {epoch} Loss {loss:.4f}")

    return params, history

In [ ]:
# Train with all optimizers
params_sgd, hist_sgd = train_cnn_optimizer(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    optimizer="sgd"
)

params_mom, hist_mom = train_cnn_optimizer(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    optimizer="momentum"
)

params_adam, hist_adam = train_cnn_optimizer(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    optimizer="adam"
)

In [ ]:
# SGD
train_sgd, val_sgd, test_sgd = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_sgd,
    cnn_forward
)

# Momentum
train_mom, val_mom, test_mom = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_mom,
    cnn_forward
)

# Adam
train_adam, val_adam, test_adam = evaluate_model(
    X_train_img, y_train_img,
    X_val_img, y_val_img,
    X_test_img, y_test_img,
    params_adam,
    cnn_forward
)

In [ ]:
print("OPTIMIZER COMPARISON")
print("----------------------------------------------")
print("Optimizer | Train | Val | Test")
print("----------------------------------------------")
print(f"SGD       | {train_sgd:.3f} | {val_sgd:.3f} | {test_sgd:.3f}")
print(f"Momentum  | {train_mom:.3f} | {val_mom:.3f} | {test_mom:.3f}")
print(f"Adam      | {train_adam:.3f} | {val_adam:.3f} | {test_adam:.3f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(history, filename="loss_plot.png", title="Loss Curve"):
    plt.figure(figsize=(6,4))
    plt.plot(history["train_loss"], linewidth=2)
    plt.plot(history["val_loss"], linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend(["Train Loss","Validation Loss"])
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()

    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_loss(hist_sgd, "sgd_loss.png", "SGD Loss Curve")
plot_loss(hist_mom, "momentum_loss.png", "Momentum Loss Curve")
plot_loss(hist_adam, "adam_loss.png", "Adam Loss Curve")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(hist_sgd["train_loss"])
plt.plot(hist_mom["train_loss"])
plt.plot(hist_adam["train_loss"])

plt.legend(["SGD","Momentum","Adam"])
plt.title("Optimizer Convergence Comparison")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig("optimizer_convergence.png", dpi=300)  # ✅ Save
plt.show()

In [ ]:
models = ["SGD", "Momentum", "Adam"]
test_acc = [test_sgd, test_mom, test_adam]

plt.figure(figsize=(6,4))
plt.bar(models, test_acc)
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy Comparison (Optimizers)")
plt.ylim(0,1)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig("optimizer_test_accuracy.png", dpi=300)  # ✅ Save
plt.show()